In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import dataset
import network
import train_network

In [ ]:
device = 'cpu'
network_shape = [3, 100, 100, 100, 2]
scheduler = train_network.Scheduler()
device

In [ ]:
net = network.Network(network_shape, scheduler).to(device)
checkpoint = torch.load('network_final.pt', map_location=device)
net.load_state_dict(checkpoint['weights'])

In [ ]:
dataset_pts = dataset.generate_initial_datapoints(10000)
plt.scatter(dataset_pts[:,0],dataset_pts[:,1])
plt.show()

In [ ]:
def trajectory(initial_line, sigma_scale):
    n = initial_line.shape[0]
    traj = np.zeros((n,2,scheduler.T+1))
    traj[:,:,-1] = initial_line
    
    with torch.no_grad():
        xs = torch.tensor(initial_line, dtype=torch.float32).to(device)
        for t in range(scheduler.T-1,-1,-1):
            xs = net.reverse_diffuse(xs, t, sigma_scale)
            traj[:, :, t] = xs.detach().cpu().numpy()
    return traj

def plot_trajectory(traj,plot_trail = True):
    times = [1000,500,100,0]
    
    fig, axes = plt.subplots(1, len(times), figsize=(16, 4))
    
    for i, ax in enumerate(axes):
        ax.scatter(dataset_pts[:,0],dataset_pts[:,1],s=1,color='grey')
        # ax.plot(traj[:,0,times[i]],traj[:,1,times[i]],color = 'Blue')
        ax.scatter(traj[:,0,times[i]],traj[:,1,times[i]],color = 'red',s=1)
        ax.set_ylim([-3, 3])
        ax.set_xlim([-3, 3])
        ax.set_aspect('equal')
        ax.set_title(f"Time {times[i]}")
        
    if plot_trail:
        for i, ax in enumerate(axes):
            for k in range(traj.shape[0]):
                ax.plot(traj[k,0,times[i]:],traj[k,1,times[i]:],c='blue',alpha=0.025)
    
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
n=100
traj1 = trajectory(np.transpose([np.linspace(0,2.5,n),np.linspace(1.5,0.5,n)]),0)
fig = plot_trajectory(traj1)
fig.savefig("interpolation1.jpg")

In [ ]:
n=100
traj2 = trajectory(np.transpose([np.linspace(-2,0,n),np.linspace(-2,-1.5,n)]),0)
fig = plot_trajectory(traj2)
fig.savefig("interpolation2.jpg")

In [ ]:
n=200
traj3 = trajectory(np.transpose([np.linspace(-2,2,n),np.linspace(2,-2,n)]),0)
fig = plot_trajectory(traj3)
fig.savefig("interpolation3.jpg")

In [ ]:
n=500
traj4 = trajectory(np.transpose([np.linspace(-2,0,n),np.linspace(-1,2,n)]),0)
fig = plot_trajectory(traj4)
fig.savefig("interpolation4.jpg")